# DoRA for Competition Math — Colab Notebook
**CS 4782 · Cornell · Spring 2026** &nbsp;·&nbsp; Ryan Ye · Boaz Ng · Leon Jiao · Aadi Singla

Reproducing *DoRA: Weight-Decomposed Low-Rank Adaptation* (Liu et al., ICML 2024) on AMC12 competition math.

**Base model:** `Qwen/Qwen2.5-3B` (base) or `Qwen/Qwen2.5-3B-Instruct`  
Qwen base weights are ungated — no HuggingFace access approval required.

**Quick-start:**
1. Run **§ 0 Setup** (install, GPU, HF auth)
2. Edit **`CFG`** in § 1 for your experiment
3. Run a single experiment cell in **§ 4**
4. Compare everything in **§ 5 Results**

---
| Section | Contents |
|---------|----------|
| § 0 | Install · GPU check · HF auth |
| § 1 | `CFG` — one place for all settings |
| § 2 | Helper functions (train / eval / results) |
| § 3 | Data stats & preview |
| § 4 | Experiments: E3 zero-shot · E1 LoRA · E2 DoRA · rank sweep · scoring comparison · heavy train · MATH bench · s1k |
| § 5 | Results comparison table |

In [ ]:
# ── download eval data (run once; skip if data/ already populated) ──────────
# !python code/download_amc12.py    # → data/amc12_2024/ and data/amc12_2025/
# !python code/download_math.py     # → data/math_l135/math_l135.jsonl
# !python code/download_s1k.py      # → data/train_s1k/train_s1k.jsonl
#
# ── build training corpora via code/data_clean/ ──────────────────────────────
# data_clean/build.py calls data_clean/sources.py (HF loaders) and
# data_clean/leakage.py (4-stage AIME contamination filter)
# !python -m code.data_clean.build --tier train_light   # 7.5K rows
# !python -m code.data_clean.build --tier train_heavy   # 27.5K rows
# !python -m code.data_clean.build --tier train_scale   # deferred — downloads OMI-2
#
# Inspect leakage report after any build:
# import json; print(json.load(open('data/train/leakage_report.json')))

---
## § 0 — Setup

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Code lives on Drive — no git clone needed.
    # Upload cs4782-project/ (everything not in .gitignore) to MyDrive/ first.
    REPO = '/content/drive/MyDrive/cs4782-project'
    if not os.path.isdir(REPO):
        raise RuntimeError(
            'Repo not found on Drive.\n'
            'Upload cs4782-project/ to MyDrive/ first.\n'
            'Expected path: /content/drive/MyDrive/cs4782-project/'
        )
    os.chdir(REPO)
else:
    # Local: run from project root
    pass

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('CWD:', ROOT)

In [ ]:
# Install dependencies (once per Colab session)
# Uncomment the first time:
# !pip install -q -r requirements.txt

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    vram_gb = float(gpu_info.split(',')[1].replace(' MiB','').strip()) / 1024
    print(f'GPU: {gpu_info}')
    print(f'  => use load_in_4bit=True if VRAM <= 16 GB (T4)  |  bf16 ok above 20 GB')
else:
    print('No GPU detected. Training will be very slow.')

In [ ]:
# HuggingFace login
# Qwen2.5-3B (base) is public — no token needed.
# Qwen2.5-3B-Instruct is also public, but login avoids rate limits.
# Store your token in Colab Secrets as HF_TOKEN, or set the env var locally.
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN set — fine for Qwen base/instruct (both public).')

In [ ]:
# Core imports (reload after code edits without restarting runtime)
import importlib, json, glob
from argparse import Namespace
from pathlib import Path

import yaml
import pandas as pd

import code.train as _train_mod
import code.eval  as _eval_mod

print('Imports OK')

---
## § 1 — Config

**Edit `CFG` here.** All experiment cells read from this dict.  
`apply_config()` writes it to `code/configs/run_config.yaml` before each run.

To run instruct experiments: set `model_name = 'Qwen/Qwen2.5-3B-Instruct'`.

In [ ]:
CFG = {
    # ── Model ──────────────────────────────────────────────────
    'model_name'  : 'Qwen/Qwen2.5-3B',          # base model (ungated)
    # instruct variant: set model_name='Qwen/Qwen2.5-3B-Instruct'
    'load_in_4bit': False,   # 3B fits in bf16 on A100/H100; set True for T4

    # ── LoRA / DoRA ────────────────────────────────────────────
    'rank'           : 8,
    'alpha'          : 32,   # 4 * rank
    'dropout'        : 0.05,
    'target_modules' : ['q_proj', 'k_proj', 'v_proj', 'up_proj', 'down_proj'],

    # ── Training ───────────────────────────────────────────────
    'lr'                   : 1e-4,
    'epochs'               : 1,
    'batch_size'           : 16,
    'per_device_batch_size': 1,
    'warmup_ratio'         : 0.03,
    'max_seq_len'          : 4096,
    'seed'                 : 42,
    'tier'                 : 'train_light',   # 7.5K MATH-lighteval examples

    # ── Checkpointing ──────────────────────────────────────────
    'save_strategy'    : 'steps',
    'save_steps'       : 100,
    'save_total_limit' : 5,

    # ── Paths (Drive setup cell below overrides these on Colab) ─
    'output_dir'  : 'results/checkpoints',
}

In [ ]:
# ── Checkpoint output directory ─────────────────────────────────────────────
# The repo (code, data, results) already lives on Drive at REPO.
# Checkpoints are large (1-3 GB each) and go to a separate Drive folder
# so they don't clutter the tracked repo structure.

if IN_COLAB:
    CKPT_DIR = '/content/drive/MyDrive/cs4782-checkpoints'
    os.makedirs(CKPT_DIR, exist_ok=True)
    CFG['output_dir'] = CKPT_DIR
    print(f'Working from:  {REPO}')
    print(f'Checkpoints → {CKPT_DIR}')
    print(f'Results     → {REPO}/results/')
else:
    print('Local mode — using local paths from CFG.')

---
## § 2 — Helper functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

def apply_config(overrides=None, path='code/configs/run_config.yaml'):
    """Merge CFG + overrides into a YAML file that train/eval scripts read."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    merged = {**CFG, **(overrides or {})}
    with open(path, 'w') as f:
        yaml.dump(merged, f, default_flow_style=False)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────────────────────────────────────

def run_train(method, tier=None, overrides=None):
    """
    Train a model adapter.

    method    : 'lora' | 'peft_dora' | 'dora'
    tier      : 'train_light' | 'train_heavy' | 'train_s1k'
    overrides : dict of CFG keys to change for this run only
    """
    config_path = apply_config(overrides)
    importlib.reload(_train_mod)
    args = Namespace(
        config=config_path,
        model=None,
        method=method,
        tier=tier or CFG['tier'],
        output_dir=CFG['output_dir'],
        resume=None,
    )
    rank_val = (overrides or {}).get('rank', CFG['rank'])
    print(f'[TRAIN] method={method}  tier={args.tier}  rank={rank_val}')
    _train_mod.main(args)


# ─────────────────────────────────────────────────────────────────────────────
# Evaluation
# ─────────────────────────────────────────────────────────────────────────────

def run_eval(checkpoint, benchmarks=('aime2024', 'aime2025'),
             n_samples=1, temperature=0.0, output=None):
    """
    Evaluate a checkpoint on AIME.

    checkpoint : local peft dir  OR  HF model name (zero-shot)
    benchmarks : 'aime2024' | 'aime2025' | ('aime2024', 'aime2025')
    n_samples  : 1 = greedy;  >1 = majority-vote at given temperature
    """
    importlib.reload(_eval_mod)
    if isinstance(benchmarks, str):
        benchmarks = (benchmarks,)
    for bench in benchmarks:
        args = Namespace(
            checkpoint=checkpoint,
            benchmark=bench,
            n_samples=n_samples,
            temperature=temperature,
            model=None,
            load_in_4bit=CFG.get('load_in_4bit', False),
            output=output,
        )
        name = Path(checkpoint).name if Path(checkpoint).exists() else checkpoint
        print(f'[EVAL]  {name}  /  {bench}')
        _eval_mod.main(args)


# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint + results inspection
# ─────────────────────────────────────────────────────────────────────────────

def list_checkpoints(output_dir=None):
    """Print a table of all saved checkpoints and return their paths."""
    d = output_dir or CFG['output_dir']
    rows = []
    for ckpt in sorted(glob.glob(os.path.join(d, '*'))):
        rc_path = os.path.join(ckpt, 'run_config.json')
        rc = json.load(open(rc_path)) if os.path.exists(rc_path) else {}
        rows.append({
            'path'  : ckpt,
            'name'  : os.path.basename(ckpt),
            'method': rc.get('method', '?'),
            'tier'  : rc.get('tier', '?'),
            'rank'  : rc.get('rank', '?'),
        })
    if rows:
        display(pd.DataFrame(rows).drop(columns=['path']))
    else:
        print('No checkpoints yet in', d)
    return [r['path'] for r in rows]


def latest_checkpoint(method_substr):
    """Return the most recent checkpoint whose name contains method_substr."""
    ckpts = sorted(glob.glob(os.path.join(CFG['output_dir'], f'*{method_substr}*')))
    if not ckpts:
        raise FileNotFoundError(f'No checkpoint matching *{method_substr}* in {CFG["output_dir"]}')
    return ckpts[-1]


def show_results(results_dir=None):
    """Load all eval JSON files and display a sorted comparison table."""
    d = results_dir or os.path.join('results', 'tables')
    rows = []
    for p in sorted(glob.glob(os.path.join(d, '*.json'))):
        r = json.load(open(p))
        rows.append({
            'checkpoint': Path(r.get('checkpoint', '')).name,
            'benchmark' : r.get('benchmark', ''),
            'accuracy'  : f"{r.get('accuracy', 0):.1%}",
            'correct'   : f"{r.get('n_correct', 0)}/{r.get('n_total', 0)}",
            'n_samples' : r.get('n_samples', 1),
        })
    if rows:
        df = (pd.DataFrame(rows)
                .sort_values(['benchmark', 'accuracy'], ascending=[True, False])
                .reset_index(drop=True))
        display(df)
    else:
        print('No results yet — run evaluation cells first.')
    return rows


print('Helpers loaded.')

In [ ]:
# ── added later: AMC12 eval, logit/two-pass scoring modes, resume support ──
# Updated run_eval defaults to AMC12 (50 problems vs 30 for AIME — cleaner signal)
# and adds a scoring parameter for the three-mode eval protocol.

def run_eval(checkpoint, benchmarks=('amc122024', 'amc122025'),
             scoring='greedy', n_samples=1, temperature=0.0, output=None):
    """
    Evaluate a checkpoint.

    checkpoint : local peft/dora dir  OR  HF model name (zero-shot)
    benchmarks : 'amc122024' | 'amc122025' | 'aime2024' | 'aime2025' | 'math' | tuple
    scoring    : 'greedy' | 'logit' | 'twopass'
                 greedy   = generate full response, parse \\boxed{}
                 logit    = single forward pass, argmax over A-E tokens (AMC12 only)
                 twopass  = generate reasoning chain, then re-read answer logits
    n_samples  : 1 = greedy;  >1 = majority-vote sampling
    """
    importlib.reload(_eval_mod)
    if isinstance(benchmarks, str):
        benchmarks = (benchmarks,)
    for bench in benchmarks:
        args = Namespace(
            checkpoint=checkpoint,
            benchmark=bench,
            scoring=scoring,
            n_samples=n_samples,
            temperature=temperature,
            model=None,
            load_in_4bit=CFG.get('load_in_4bit', False),
            output=output,
        )
        name = Path(checkpoint).name if Path(checkpoint).exists() else checkpoint
        print(f'[EVAL]  {name}  /  {bench}  /  {scoring}')
        _eval_mod.main(args)


def run_train(method, tier=None, overrides=None, resume=None):
    """
    Train a model adapter (updated: added resume support for preempted jobs).

    resume : path to a checkpoint dir to resume from, e.g.
             '.../cs4782-dora/checkpoints/lora_train_0501/checkpoint-400'
    """
    config_path = apply_config(overrides)
    importlib.reload(_train_mod)
    args = Namespace(
        config=config_path,
        model=None,
        method=method,
        tier=tier or CFG['tier'],
        output_dir=CFG['output_dir'],
        resume=resume,
    )
    rank_val = (overrides or {}).get('rank', CFG['rank'])
    print(f'[TRAIN] method={method}  tier={args.tier}  rank={rank_val}'
          + (f'  resume={resume}' if resume else ''))
    _train_mod.main(args)


def run_eval_math(checkpoint):
    """Evaluate on MATH benchmark (levels 1/3/5, pre-filtered in math_l135.jsonl)."""
    importlib.reload(_eval_mod)
    args = Namespace(
        checkpoint=checkpoint,
        benchmark='math',
        scoring='greedy',
        n_samples=1,
        temperature=0.0,
        model=None,
        load_in_4bit=CFG.get('load_in_4bit', False),
        output=None,
    )
    name = Path(checkpoint).name if Path(checkpoint).exists() else checkpoint
    print(f'[EVAL]  {name}  /  math  levels={levels}')
    _eval_mod.main(args)


print('Extended helpers loaded.')

---
## § 3 — Data

Verify training data and preview eval problems.  
If a tier is missing, build it: `!python -m code.data_clean.build --tier train_light`

In [ ]:
from code.data import load_aime, load_train_corpus

# ── Training corpus stats ──────────────────────────────────────────────────
for tier in ['train_light', 'train_heavy']:
    try:
        corpus = load_train_corpus(tier)
        srcs = {}
        for row in corpus:
            s = row.get('source', 'unknown')
            srcs[s] = srcs.get(s, 0) + 1
        print(f'{tier:14s}: {len(corpus):>6,} rows  |  {srcs}')
    except FileNotFoundError:
        print(f'{tier:14s}: NOT BUILT  -> python -m code.data_clean.build --tier {tier}')

print()

# ── AIME preview ──────────────────────────────────────────────────────────
for year in [2024, 2025]:
    problems = load_aime(year)
    p0 = problems[0]
    print(f'AIME {year}: {len(problems)} problems')
    print(f'  [{p0["id"]}]  answer={p0["answer"]}')
    print(f'  {p0["problem"][:120]}...')
    print()

# ── AMC12 preview ─────────────────────────────────────────────────────────
import json, glob as _g
for year in [2024, 2025]:
    files = sorted(_g.glob(f'data/amc12_{year}/*.json'))
    p0 = json.load(open(files[0]))
    print(f'AMC12 {year}: {len(files)} problems')
    print(f'  answer={p0["answer"]}  difficulty={p0.get("difficulty","?")}')
    print(f'  {p0["problem"][:120]}...')
    print()

---
## § 4 — Experiments

Each cell is **independent** — run any one without running the others.

| Cell | Experiment | Trains? | Notes |
|------|-----------|---------|-------|
| E3 | Zero-shot Qwen baseline | No | Eval on AMC12 + AIME, no adapter |
| E1 | LoRA | Yes | Primary comparison baseline |
| E2 | DoRA | Yes | Our custom DoRA implementation |
| Sanity | DoRA vs Peft DoRA | Yes | Verify our implementation matches reference |
| Rank sweep | LoRA + DoRA at r ∈ {2, 4, 8} | Yes | Logit eval on AMC12 |
| Scoring | Scoring mode comparison | No | Logit / two-pass / greedy on same checkpoint |
| Heavy | Heavy training tier | Yes | train_heavy (27.5K rows) |
| MATH | MATH benchmark | No | Greedy eval, levels 1/3/5 |
| s1k | S1K experiment | Yes | 1K rows, r=32, 5 epochs |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  E3 — Zero-shot baseline  (no fine-tuning, accuracy floor)      ║
# ╚══════════════════════════════════════════════════════════════════╝
run_eval(CFG['model_name'], benchmarks=('amc122024', 'amc122025', 'aime2024'), scoring='greedy')
run_eval(CFG['model_name'], benchmarks=('amc122024',), scoring='logit')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  E1 — LoRA  (primary comparison baseline)                       ║
# ╚══════════════════════════════════════════════════════════════════╝
run_train('lora')
run_eval(latest_checkpoint('lora_'), benchmarks=('amc122024', 'amc122025'), scoring='greedy')
run_eval(latest_checkpoint('lora_'), benchmarks=('amc122024',), scoring='logit')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  E2 — DoRA  (our custom implementation)                         ║
# ╚══════════════════════════════════════════════════════════════════╝
run_train('dora')
run_eval(latest_checkpoint('dora_'), benchmarks=('amc122024', 'amc122025'), scoring='greedy')
run_eval(latest_checkpoint('dora_'), benchmarks=('amc122024',), scoring='logit')

In [ ]:
# ── sanity check: DoRA (ours) vs Peft DoRA (reference) ──
# Run each once at identical config — logit scores should be within noise.
run_train('dora',      overrides={'epochs': 1, 'rank': 8, 'alpha': 32})
run_train('peft_dora', overrides={'epochs': 1, 'rank': 8, 'alpha': 32})

print('\n── DoRA (ours) ──')
run_eval(latest_checkpoint('dora_'),      benchmarks='amc122024', scoring='logit')

print('\n── Peft DoRA ──')
run_eval(latest_checkpoint('peft_dora_'), benchmarks='amc122024', scoring='logit')

show_results()

In [ ]:
# ── rank sweep: LoRA + DoRA at r ∈ {2, 4, 8} ────────────────────────────
# AMC12 has more problems than AIME (50 vs 30) — logit eval gives cleaner signal
# alpha = 4 * rank throughout, consistent with paper Table 8

SWEEP_RANKS = [2, 4, 8]

for r in SWEEP_RANKS:
    print(f'\n=== LoRA rank {r} ===')
    run_train('lora', overrides={'rank': r, 'alpha': 4 * r, 'epochs': 3, 'lr': 5e-5})

for r in SWEEP_RANKS:
    print(f'\n=== DoRA rank {r} ===')
    run_train('dora', overrides={'rank': r, 'alpha': 4 * r, 'epochs': 3, 'lr': 5e-5})

# Eval all sweep checkpoints with logit scoring on AMC12 2024
for ckpt in list_checkpoints():
    rc_path = os.path.join(ckpt, 'run_config.json')
    if os.path.exists(rc_path):
        rc = json.load(open(rc_path))
        if rc.get('rank') in SWEEP_RANKS:
            run_eval(ckpt, benchmarks='amc122024', scoring='logit')

In [ ]:
# ── scoring mode comparison: same checkpoint, three modes ──
# Shows the logit-greedy gap: model may 'know' the answer but fail to say it
ckpt = latest_checkpoint('lora_')
for scoring in ['logit', 'twopass', 'greedy']:
    run_eval(ckpt, benchmarks='amc122024', scoring=scoring)
print('\n-- DoRA --')
ckpt_dora = latest_checkpoint('dora_')
for scoring in ['logit', 'twopass', 'greedy']:
    run_eval(ckpt_dora, benchmarks='amc122024', scoring=scoring)
show_results()

In [ ]:
# ── heavy training: train_heavy tier (27.5K rows) ──
# train_light is MATH-lighteval only; train_heavy adds NuminaMath-CoT
# NuminaMath is a CoT dataset with olympiad/proof problems — may introduce
# different reasoning patterns beyond pure AMC12-style computation
run_train('lora', tier='train_heavy', overrides={'epochs': 1, 'lr': 1e-4})
run_train('dora', tier='train_heavy', overrides={'epochs': 1, 'lr': 1e-4})

run_eval(latest_checkpoint('lora_train_heavy'),
         benchmarks=('amc122024', 'amc122025'), scoring='greedy')
run_eval(latest_checkpoint('dora_train_heavy'),
         benchmarks=('amc122024', 'amc122025'), scoring='greedy')
run_eval(latest_checkpoint('dora_train_heavy'),
         benchmarks=('amc122024',), scoring='logit')

In [ ]:
%%time
# ── MATH benchmark: 723 problems, levels 1 / 3 / 5 ──
# Measures general math reasoning; run on key checkpoints as a regression check
# Full eval is slow (~40 min per checkpoint) — run overnight or in background
for ckpt_pattern in ['lora_train_light', 'dora_train_light', 'lora_train_heavy']:
    try:
        ckpt = latest_checkpoint(ckpt_pattern)
        run_eval_math(ckpt)
    except FileNotFoundError:
        print(f'No checkpoint for {ckpt_pattern}')

In [ ]:
# ── s1k experiment: 1K-problem subset, r=32, 5 epochs ──
# Tests whether a small high-quality dataset can match large-data training
# (motivated by the S1 paper: "simple scalable test-time compute")
run_train('lora', tier='train_s1k',
          overrides={'rank': 32, 'alpha': 128, 'epochs': 5, 'lr': 1e-5,
                     'warmup_ratio': 0.05, 'max_seq_len': 8192,
                     'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']})
run_eval(latest_checkpoint('lora_s1k'), benchmarks=('amc122024', 'amc122025'), scoring='greedy')
run_eval_math(latest_checkpoint('lora_s1k'))

---
## § 5 — Results

In [ ]:
# All experiments side-by-side
show_results()

In [ ]:
# Checkpoint inventory
list_checkpoints()

In [ ]:
# ── regenerate publication figures ────────────────────────────────────────────
# Reads results/tables/*.json output from eval runs; writes 4 PNGs to results/figures/
# !python code/plot_figures.py --out-dir results/figures
# !python code/plot_results.py --out-dir results/figures

In [ ]:
# ── sweep and flat-result analysis ────────────────────────────────────────────
# Compare all flat eval results side-by-side:
# !python code/compare_results.py --benchmark amc122024
# Checkpoint-by-checkpoint accuracy tables:
# !python code/compare_sweep.py --run lora_train_light
# !python code/analyze_sweep.py --run-id dora_train_light --csv results/sweep_summary.csv

In [ ]:
# ── Final results summary (AMC12 2024) ──
# Numbers from our full HPC runs (not re-run here — for display purposes)
print("""
AMC12 2024 — logit / greedy / Δ
  Base (3B-base)       logit=32%   greedy= —    Δ=  —
  Base (3B-instruct)   logit=34%   greedy=44%   Δ=−10pp
  LoRA  light (base)   logit=38%   greedy=18%   Δ=+20pp
  LoRA  light (inst)   logit=44%   greedy=10%   Δ=+34pp  ← best logit (tie)
  LoRA  heavy (base)   logit=34%   greedy=36%   Δ= −2pp
  DoRA  light (base)   logit=38%   greedy=12%   Δ=+26pp
  DoRA  light (inst)   logit=36%   greedy= 8%   Δ=+28pp
  DoRA  heavy (base)   logit=32%   greedy=46%   Δ=−14pp  ← best greedy
  DoRA  r4    (base)   logit=44%   greedy=16%   Δ=+28pp  ← best logit (tie)

MATH benchmark (greedy, levels 1/3/5)
  Baseline (instruct)  overall=52.7%  L1=82.6%  L3=66.1%  L5=31.4%
  LoRA s1k (inst)      overall=44.8%  L1=77.1%  L3=56.5%  L5=24.2%
  LoRA heavy (base)    overall=35.0%  L1=73.4%  L3=43.5%  L5=15.1%
  DoRA light (base)    overall=30.7%  L1=70.6%  L3=36.4%  L5=12.7%
""")

---
## § 6 — One-off utilities

Cells for common ad-hoc tasks.

In [ ]:
# ── average adapter checkpoints (often improves eval accuracy) ────────────────
# !python code/avg_checkpoints.py --run_dir results/checkpoints/lora_train_light
# !python code/avg_checkpoints.py --run_dir results/checkpoints/dora_train_light --steps 700 1100 1407

In [ ]:
# ── rescore saved responses with updated answer extraction ────────────────────
# !python code/rescore.py
#
# ── merge parallel MATH eval shards (if run across multiple jobs) ──────────────
# !python code/merge_math_shards.py --run_id dora_train_light --num_shards 4

In [ ]:
# ── Eval a specific checkpoint manually ──────────────────────────────────
# ckpt = 'results/checkpoints/lora_train_MMDD_HHMM'
# run_eval(ckpt, benchmarks='amc122025', scoring='logit')
# run_eval(ckpt, benchmarks='aime2025', n_samples=16, temperature=0.7)

In [ ]:
# ── Build training data ─────────────────────────────────────────────────
# !python -m code.data_clean.build --tier train_light
# !python -m code.data_clean.build --tier train_heavy
# !python -m code.data_clean.build --tier train_s1k

In [ ]:
# ── LR sweep helper ──────────────────────────────────────────────────────
# Quick sanity check before committing to a full training run
# for lr in [5e-5, 1e-4, 2e-4]:
#     run_train('lora', tier='train_light', overrides={'lr': lr, 'epochs': 1})
# show_results()

In [ ]:
# ── DoRALinear unit tests (CPU, ~10 s) ──
!python -m pytest tests/test_dora_grad.py -v --tb=short 2>&1